<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 1</b>

## Unit 5: Advanced RAG Techniques

**CV Raman Global University, Bhubaneswar**  
*AI Center of Excellence*

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Poorit-Technologies/cvraman-coe/blob/main/courses-contents/ai-systems-engineering-1/unit-5/02-ai-systems-engineering-1-unit5-advanced-rag.ipynb)

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Implement reranking** - reorder retrieved documents with LLM
2. **Query rewriting** - transform user questions for better retrieval
3. **Combine techniques** for improved RAG accuracy
4. **Compare results** before and after optimization

**Duration:** ~1.5 hours

---

## 1. Environment Setup

In [ ]:
# Setup
!pip install -q openai chromadb sentence-transformers

import os
import json
from getpass import getpass
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb

api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
client = OpenAI(api_key=api_key)

MODEL = "gpt-4o-mini"

---

## 2. Advanced RAG Architecture

Basic RAG assumes that vector similarity equals relevance — the closest embeddings must be the best context. In practice, this breaks down when:

1. **Embeddings miss nuance** — a document about "PhD research at MIT" may not rank highest for "Who has advanced degrees?" because embedding spaces don't perfectly capture semantic equivalence.
2. **Queries are vague or conversational** — "What do I get if I join?" is a natural question, but a poor search query for finding benefits documentation.

Advanced RAG adds two stages to address this: **query rewriting** (before retrieval) and **reranking** (after retrieval).

```
┌─────────────────────────────────────────────────────────────────┐
│                    Advanced RAG Pipeline                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  [User Query] → [Query Rewriting] → [Better Query]              │
│                                           ↓                     │
│                                   [Vector Search]               │
│                                           ↓                     │
│                     [Retrieved Docs] → [Reranking] → [Top Docs] │
│                                                          ↓      │
│                              [Context + Query] → [LLM] → [Answer]│
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Comparison

| Stage | Basic RAG | Advanced RAG |
|-------|-----------|-------------- |
| Query | Used as-is | Rewritten for clarity |
| Retrieval | Vector similarity | Vector similarity |
| Ranking | Embedding distance only | LLM-based reranking |
| Context | Top-k by distance | Top-k by relevance |

> **Reference:** [Retrieval-Augmented Generation for Large Language Models: A Survey (Gao et al., 2024)](https://arxiv.org/abs/2312.10997) provides a comprehensive taxonomy of RAG enhancements including query transformation and reranking.

---

## 3. Knowledge Base and Retrieval

We'll use a ChromaDB-backed knowledge base — a collection of documents about a fictional company, TechSolutions India. The `retrieve()` function performs semantic search: it encodes the query, finds the closest document embeddings, and returns the top results.

In [ ]:
# Knowledge base documents
knowledge_documents = [
    # Company
    "TechSolutions India was founded in 2018 by Priya Sharma and Rahul Verma in Bhubaneswar, Odisha. The company has grown to 250+ employees and achieved ₹50 crores revenue in 2024.",
    "TechSolutions has offices in Bhubaneswar (headquarters), Bangalore, and Hyderabad. The company specializes in AI/ML solutions, cloud services, and mobile applications.",

    # Leadership
    "Priya Sharma is the CEO and co-founder. Education: IIT Delhi (B.Tech), Stanford (MBA). Previous: Senior Director at Infosys. Awards: Women in Tech Leader 2022.",
    "Rahul Verma is the CTO and co-founder. Education: BITS Pilani, IIM Bangalore. Previous: Tech Lead at Google India. Expertise: Machine Learning, Cloud Architecture.",
    "Ananya Patel is VP of Engineering. She joined in 2019 and leads a team of 100+ engineers. She holds a PhD from IISc Bangalore in Computer Science.",
    "Vikram Reddy is the Head of AI Research. He completed his PhD from MIT and has published 50+ papers on deep learning and NLP.",

    # Products
    "CloudAssist Pro is the flagship enterprise cloud management platform. Price: ₹50,000/month. Features: auto-scaling, real-time monitoring, cost optimization, 24/7 support.",
    "SmartHR is an AI-powered HR management system. Price: ₹25,000/month. Features: recruitment automation, payroll processing, performance tracking, employee analytics.",
    "DataViz Analytics is a business intelligence platform. Price: ₹15,000/month. Features: real-time dashboards, custom reports, data integration, predictive analytics.",
    "MobileFirst SDK helps developers build cross-platform mobile apps. Price: ₹10,000/month. Supports iOS, Android, and web platforms.",

    # Policies
    "Work hours: 9 AM to 6 PM, Monday to Friday. Hybrid model: 3 days office, 2 days remote. Core hours are 10 AM to 4 PM when everyone should be available.",
    "Leave policy: 24 paid leaves + 10 sick leaves per year. Maternity leave: 26 weeks. Paternity leave: 2 weeks. Unused leaves can carry forward up to 10 days.",
    "Probation period is 6 months for all new employees. Notice period is 2 months for permanent employees, 1 month during probation.",

    # Benefits
    "Health insurance: ₹5 lakh coverage for employee and family. Includes hospitalization, OPD, dental, and vision. Premium fully paid by company.",
    "Learning budget: ₹50,000 per year for courses, certifications, and conferences. Access to Coursera, Udemy, and O'Reilly Learning included.",
    "Performance bonus: up to 20% of annual salary. Paid in April after annual reviews. Based on individual and company performance.",
    "Other benefits: Gym membership, meal allowance, cab facility for late working, team outings every quarter.",

    # Clients and Projects
    "Major clients: HDFC Bank, Tata Motors, Reliance Industries, ICICI Bank, Infosys, and several startups.",
    "Completed 200+ projects across banking, automotive, retail, and healthcare sectors. Customer satisfaction rate: 95%.",
    "Recent project: Built AI-powered fraud detection system for HDFC Bank, reducing fraud by 60%."
]

# Initialize components and index documents
embedder = SentenceTransformer('all-MiniLM-L6-v2')
chroma = chromadb.Client()

try:
    chroma.delete_collection("advanced_rag")
except:
    pass

collection = chroma.create_collection("advanced_rag")

embeddings = embedder.encode(knowledge_documents)
collection.add(
    ids=[f"doc_{i}" for i in range(len(knowledge_documents))],
    embeddings=embeddings.tolist(),
    documents=knowledge_documents
)

print(f"Indexed {collection.count()} documents")


def retrieve(query, n_results=5):
    """Retrieve documents from vector store."""
    query_embedding = embedder.encode([query])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )
    return results['documents'][0]

---

## 4. Reranking

**Reranking** is a second-pass relevance scoring step applied after initial retrieval. The key insight: embedding models encode documents *independently* of any query, so the similarity score reflects general semantic closeness, not direct relevance to the specific question. A reranker evaluates each **query–document pair** jointly, producing a more accurate relevance judgment.

### How It Works

1. Retrieve top-k documents using vector similarity (fast, approximate)
2. Score each document against the query using a reranker (slow, precise)
3. Reorder documents by reranker score

### Reranking Approaches

| Approach | Examples | Pros | Cons |
|----------|----------|------|------|
| **Cross-encoder models** | Cohere Rerank, `ms-marco-MiniLM` | Fast, cheap, purpose-built | Requires separate model |
| **LLM-based reranking** | GPT-4o, Claude as judge | No extra model, high quality | Slower, more expensive |

In this notebook, we use LLM-based reranking for simplicity. In production, dedicated cross-encoder models like [Cohere Rerank](https://docs.cohere.com/docs/rerank) offer better latency at lower cost.

> **Reference:** [Passage Re-ranking with BERT (Nogueira & Cho, 2019)](https://arxiv.org/abs/1901.04085) introduced the cross-encoder approach to neural reranking that underlies most modern rerankers.

In [ ]:
RERANK_PROMPT = """
You are a document re-ranker. Given a question and retrieved documents, 
reorder them from most to least relevant.

Question: {question}

Documents:
{documents}

Return a JSON array of document numbers in order of relevance.
Example: [3, 1, 5, 2, 4]

Only return the JSON array, nothing else.
"""


def rerank(question, documents):
    """Rerank documents using LLM."""
    docs_text = ""
    for i, doc in enumerate(documents, 1):
        docs_text += f"\n[Document {i}]:\n{doc}\n"

    prompt = RERANK_PROMPT.format(question=question, documents=docs_text)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    order_text = response.choices[0].message.content.strip()
    order = json.loads(order_text)

    reranked = [documents[i - 1] for i in order if 1 <= i <= len(documents)]
    return reranked, order


# Demo: reranking in action
query = "Who has a PhD?"
docs = retrieve(query, n_results=5)

print(f"Query: {query}\n")
print("Before reranking (top 3):")
for i, doc in enumerate(docs[:3], 1):
    print(f"  {i}. {doc[:80]}...")

reranked_docs, order = rerank(query, docs)
print(f"\nRerank order: {order}\n")
print("After reranking (top 3):")
for i, doc in enumerate(reranked_docs[:3], 1):
    print(f"  {i}. {doc[:80]}...")

---

## 5. Query Rewriting

**Query rewriting** transforms a user's natural-language question into a more effective search query *before* retrieval. This addresses two common problems:

1. **Vague queries** — Users often ask casual questions like *"What do I get if I join?"* when they mean *"employee benefits and compensation packages."* The original phrasing may not match the vocabulary in your documents.

2. **Conversational follow-ups** — In multi-turn conversations, users rely on coreference (*"Where did she study?"*) that makes no sense without prior context. Query rewriting resolves these references into self-contained queries.

### Example

| User says | Rewritten query |
|-----------|----------------|
| "What do I get if I join?" | "employee benefits compensation TechSolutions" |
| "Tell me about the big boss" | "CEO founder TechSolutions India leadership" |
| (after asking about Priya) "Where did she study?" | "Priya Sharma education university degree" |

> **References:**
> - [Query Rewriting for Retrieval-Augmented Large Language Models (Ma et al., 2023)](https://arxiv.org/abs/2305.14283)
> - [Query Transformations (LangChain Blog)](https://blog.langchain.dev/query-transformations/)

In [ ]:
REWRITE_PROMPT = """
You are helping improve a search query for a knowledge base about TechSolutions India.

Conversation history:
{history}

Current question: {question}

Rewrite this as a clear, specific search query that will find the most relevant documents.
Focus on key terms and entities.
Return ONLY the rewritten query, nothing else.
"""


def rewrite_query(question, history=[]):
    """Rewrite query for better retrieval."""
    history_text = "\n".join([f"{m['role']}: {m['content']}" for m in history]) if history else "None"

    prompt = REWRITE_PROMPT.format(history=history_text, question=question)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip()


# Demo: query rewriting
vague_queries = [
    "What do I get if I join?",
    "Tell me about the big boss",
    "How much money can I make extra?"
]

print("Query Rewriting Examples:\n")
for q in vague_queries:
    rewritten = rewrite_query(q)
    print(f"  Original:  {q}")
    print(f"  Rewritten: {rewritten}\n")

# With conversation history (coreference resolution)
history = [
    {"role": "user", "content": "Tell me about Priya"},
    {"role": "assistant", "content": "Priya Sharma is the CEO of TechSolutions India."}
]

follow_up = "Where did she study?"
rewritten = rewrite_query(follow_up, history)
print(f"With history (user asked about Priya):")
print(f"  Follow-up: {follow_up}")
print(f"  Rewritten: {rewritten}")

---

## 6. Advanced RAG Pipeline

The full pipeline chains all components: **rewrite → retrieve → rerank → generate**.

```
User Query → Rewrite Query → Retrieve 5 docs → Rerank → Top 3 → LLM → Answer
```

We retrieve 5 documents but only pass the top 3 (after reranking) to the LLM. This balances **recall** (casting a wide net) with **precision** (keeping context focused). Research shows that LLMs perform worse when relevant information is buried in the middle of long contexts.

> **Reference:** [Lost in the Middle: How Language Models Use Long Contexts (Liu et al., 2023)](https://arxiv.org/abs/2307.03172) demonstrates that LLM performance degrades when key information appears in the middle of the context window.

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant for TechSolutions India.
Answer questions based on the provided context.
Be accurate and concise. If the answer is not in the context, say so.

Context:
{context}
"""


def advanced_rag(question, history=[], use_rewrite=True, use_rerank=True):
    """Advanced RAG with query rewriting and reranking."""

    # Step 1: Query rewriting
    search_query = rewrite_query(question, history) if use_rewrite else question

    # Step 2: Retrieve documents
    docs = retrieve(search_query, n_results=5)

    # Step 3: Rerank
    if use_rerank:
        docs, _ = rerank(question, docs)

    # Step 4: Generate answer
    context = "\n\n".join(docs[:3])  # Use top 3 after reranking
    system_message = SYSTEM_PROMPT.format(context=context)

    messages = [
        {"role": "system", "content": system_message}
    ] + history + [
        {"role": "user", "content": question}
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0
    )

    return {
        "answer": response.choices[0].message.content,
        "search_query": search_query,
        "docs_used": docs[:3]
    }

---

## 7. Basic vs Advanced RAG

Let's run the same vague questions through both basic RAG (no rewriting, no reranking) and advanced RAG to see the difference. Pay attention to how query rewriting transforms each question.

In [ ]:
comparison_questions = [
    "What do I get if I join?",
    "Who went to the best schools?",
    "What's the deal with working from home?"
]

for q in comparison_questions:
    print(f"Q: {q}")

    basic = advanced_rag(q, use_rewrite=False, use_rerank=False)
    print(f"\n  Basic:    {basic['answer']}")

    advanced = advanced_rag(q)
    print(f"\n  Advanced: {advanced['answer']}")
    print(f"  (query rewritten to: {advanced['search_query']})")
    print(f"\n{'—' * 60}\n")

---

## 8. When to Use What

| Technique | Best For | Latency | API Cost | Complexity |
|-----------|----------|---------|----------|------------|
| **Basic RAG** | Specific, well-phrased queries | Low | 1 LLM call | Simple |
| **+ Reranking** | Noisy retrieval, precision-critical apps | +100–500ms | +1 LLM call | Moderate |
| **+ Query Rewriting** | Conversational UI, vague user queries | +100–300ms | +1 LLM call | Moderate |
| **Both** | Production systems, general-purpose chatbots | +200–800ms | +2 LLM calls | Higher |

**Production tip:** For reranking at scale, consider dedicated cross-encoder models (e.g., Cohere Rerank, `cross-encoder/ms-marco-MiniLM-L-6-v2`) instead of LLM-based reranking. They're faster, cheaper, and purpose-built for the task.

### Key Takeaways

1. **Embedding similarity ≠ relevance** — Reranking with a cross-encoder or LLM evaluates query–document pairs jointly, catching what embeddings miss.

2. **Users don't write search queries** — Query rewriting bridges the gap between natural conversation ("What do I get if I join?") and effective retrieval ("employee benefits compensation").

3. **Retrieve broad, use narrow** — Retrieve more documents than you need (5), rerank, then pass only the best (3) to the LLM. This balances recall and context quality.

4. **Each technique adds latency and cost** — Apply them where they matter: reranking for precision-critical answers, query rewriting for conversational interfaces.

### Resources

**Papers:**
- [Retrieval-Augmented Generation for Large Language Models: A Survey (Gao et al., 2024)](https://arxiv.org/abs/2312.10997)
- [Passage Re-ranking with BERT (Nogueira & Cho, 2019)](https://arxiv.org/abs/1901.04085)
- [Query Rewriting for Retrieval-Augmented LLMs (Ma et al., 2023)](https://arxiv.org/abs/2305.14283)
- [Lost in the Middle: How Language Models Use Long Contexts (Liu et al., 2023)](https://arxiv.org/abs/2307.03172)

**Guides:**
- [Advanced RAG Techniques (Pinecone)](https://www.pinecone.io/learn/advanced-rag-techniques/)
- [Query Transformations (LangChain Blog)](https://blog.langchain.dev/query-transformations/)
- [Cohere Rerank Documentation](https://docs.cohere.com/docs/rerank)
- [Sentence Transformers Cross-Encoders](https://www.sbert.net/docs/cross_encoder/usage/usage.html)

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 1
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---